# CS920 · Lab 2 — Learning a linear digit classifier

**50 minutes** · **NumPy + Matplotlib** · **No GPU or backpropagation needed**

In the lecture, we built an MNIST classifier by averaging images belonging to each digit. Today we keep **exactly the same linear score model**, but learn its weights by minimising **cross-entropy** using **gradient descent**.

**By the end, you should be able to:**
1. Explain the difference between a score and a cross-entropy loss.
2. Implement a gradient-descent update using a supplied gradient formula.
3. Compare an average-image classifier with a classifier learnt from data.

There are **three short coding tasks**. All other code is supplied.

### Start JupyterLab on a DCS lab desktop

Open a terminal, load the centrally managed CS920 environment **before** activating your virtual environment:

```bash
module load cs920-python
cd ~/cs920
# python -m venv venv # only if you didn't create the venv last week
source venv/bin/activate
cp -r /modules/cs920/labs/lab2 .
cd lab2
jupyter lab
```

Open `lab2.ipynb` and run the cells in order.

Today we use **the same complete MNIST training and test sets used in the lecture**: 60,000 training images and 10,000 test images, loaded through `torchvision.datasets.MNIST`. The instructor will cache them in `/modules/cs920/data` before class, so students do not need to download them. Only the **storage directory** changes from the lecture's `datasets.MNIST('.', ...)` calls; the underlying datasets are identical.

## 1. The model from the lecture (about 10 minutes)

Each MNIST image is $28\times28$ pixels. We flatten its pixel values into a vector $x\in\mathbb R^{784}$, with values between $0$ and $1$.

For each digit $j\in\{0,\ldots,9\}$, the model gives a **score**:

$$\boxed{S(x,j)=(Wx)_j=\langle w_j,x\rangle,\qquad W\in\mathbb R^{10\times784}.}$$

The predicted digit is the one with the **highest score**: $\hat y(x)=\arg\max_j S(x,j)$.

In the language of the lectures, we can equivalently define the **energy** $E_W(x,j)=-S(x,j)$: lower energy means greater compatibility.

**Important:** The input representation $x$ is fixed. We will learn only the matrix $W$.

In [ ]:
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt

# Exactly the same MNIST data loader as the lecture, using the shared DCS cache.
# Lecture: datasets.MNIST('.', train=True/False, download=True)
# Here:    same datasets and official splits, but pre-downloaded centrally.
mnist_train = datasets.MNIST('/modules/cs920/data', train=True, download=False)
mnist_test = datasets.MNIST('/modules/cs920/data', train=False, download=False)

# Flatten each 28x28 image to a 784-dimensional vector.
# IMPORTANT: Match the lecture's pixel scaling to reproduce its numbers.
# Here we use pixel values in [0,1] (divide raw 0-255 values by 255).
X_train = mnist_train.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_train = mnist_train.targets.numpy().astype(np.int64)
X_test = mnist_test.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_test = mnist_test.targets.numpy().astype(np.int64)

assert X_train.shape == (60000, 784) and X_test.shape == (10000, 784)
print(f'Training: {len(y_train):,} images; test: {len(y_test):,} images')
print('Pixel range:', X_train.min(), 'to', X_train.max())

fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(str(y_train[i])); ax.axis('off')
plt.suptitle('Examples from the full MNIST training set')
plt.tight_layout()
plt.show()

### The baseline: one average image for each digit

In the lecture, we constructed row $w_j$ by averaging the training images labelled $j$:

$$w_j^{\mathrm{avg}}=\frac{1}{n_j}\sum_{r:y_r=j}x_r.$$

This defines a linear classifier without minimising any classification loss.

**Task A (one line).** Complete the function computing scores for *a whole batch* of images. If `X` has shape `(n, 784)`, its scores should have shape `(n, 10)`.

In [ ]:
def scores(X, W):
    # TODO A: return all scores, shape (number of images, 10).
    return ...

# Quick self-check
W_dummy = np.zeros((10, 784), dtype=np.float32)
assert scores(X_train[:3], W_dummy).shape == (3, 10)
print('Task A passed')

In [ ]:
W_avg = np.stack([X_train[y_train == j].mean(axis=0)
                  for j in range(10)]).astype(np.float32)

def accuracy(X, y, W):
    return np.mean(np.argmax(scores(X, W), axis=1) == y)

print(f'Class-average training accuracy: {100 * accuracy(X_train, y_train, W_avg):.1f}%')

fig, axes = plt.subplots(1, 10, figsize=(12, 1.9))
for j, ax in enumerate(axes):
    ax.imshow(W_avg[j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(str(j)); ax.axis('off')
plt.suptitle('Average-image weights from the lecture (all 60,000 training images)', y=1.07)
plt.tight_layout(); plt.show()

## 2. How do we choose better weights? (about 10 minutes)

A model produces **scores**, not probabilities. We turn scores into probabilities with the **softmax** function:

$$p_W(j\mid x)=\frac{e^{S(x,j)}}{\sum_{k=0}^{9}e^{S(x,k)}}.$$

For an image whose true digit is $y$, use the **cross-entropy loss**:

$$\boxed{\ell(W;x,y)=-\log p_W(y\mid x).}$$

This loss is small when the model gives the correct digit high probability, and large when it gives the correct digit low probability. Notice that the **loss is not the energy**: it depends on *all ten class scores*.

Our goal is to minimise the **average training cross-entropy**:

$$L(W)=\frac{1}{n}\sum_{r=1}^{n}\ell(W;x_r,y_r).$$

We supply working versions of `softmax` and `cross_entropy` below, so you can concentrate on learning $W$.

In [ ]:
def softmax(S):
    # Subtract the row maximum for numerical stability.
    Z = S - S.max(axis=1, keepdims=True)
    expZ = np.exp(Z)
    return expZ / expZ.sum(axis=1, keepdims=True)

def cross_entropy(X, y, W):
    S = scores(X, W)
    Z = S - S.max(axis=1, keepdims=True)
    log_partition = np.log(np.exp(Z).sum(axis=1))
    return float(np.mean(log_partition - Z[np.arange(len(y)), y]))

# Zero weights give equal probabilities to all ten digits.
W_zero = np.zeros_like(W_avg)
print('Cross-entropy with zero weights:', round(cross_entropy(X_train[:1000], y_train[:1000], W_zero), 3))
print('Expected value: log(10) =', round(np.log(10), 3))

# Stop and predict: if the correct label has a tiny probability,
# should cross-entropy be large or small?

## 3. Learn $W$ by gradient descent (about 15 minutes)

In the next lecture, we will derive the very useful formula

$$\frac{\partial\ell}{\partial S(x,j)}=p_W(j\mid x)-\mathbf 1[j=y].$$

It says: for the **correct digit**, push its score upwards; for **incorrect digits**, push their scores downwards (more strongly when the model gives them high probability).

For all $n$ training images at once, let

- `X` be the `(n, 784)` matrix of flattened images;
- `P` be the `(n, 10)` matrix of probabilities;
- `Y` be the `(n, 10)` matrix whose row has a 1 at the correct digit and 0 elsewhere.

Then the gradient of the **average** cross-entropy is

$$\boxed{\nabla_W L(W)=\frac{1}{n}(P-Y)^\top X.}$$

You do **not** need to derive this formula or use backpropagation. Just use it in the gradient-descent update

$$\boxed{W\leftarrow W-\eta\,\nabla_W L(W),}$$

where $\eta$ is the **learning rate**. Here, one update uses **all** the training images: this is *full-batch gradient descent*, not SGD.

**Task B (two lines).** Complete the gradient computation. The code has already computed the class probabilities. Instead of constructing the entire matrix `Y`, we subtract 1 from the probability of the correct class in each row.

In [ ]:
def gradient(W, X, y):
    P = softmax(scores(X, W))      # Shape: (n, 10)
    errors = P.copy()             # Will become P - Y
    # TODO B1: subtract 1 from errors[r, y[r]] for every row r.
    # TODO B2: calculate (errors.T @ X) / n.
    return ...

# Self-check: the gradient should have the same shape as W.
assert gradient(W_zero, X_train[:20], y_train[:20]).shape == (10, 784)
print('Task B passed')

**Task C (one line).** Complete the update inside the training loop. We initialise $W$ to zero, rather than to the class averages, so that we can see what cross-entropy minimisation learns *from scratch*. We will compare the result with the class-average baseline afterward.

In [ ]:
def train(W_initial, X, y, steps=75, learning_rate=0.15):
    W = W_initial.copy()
    training_losses = []

    for step in range(steps + 1):
        if step % 5 == 0:
            training_losses.append(cross_entropy(X, y, W))
        if step == steps:
            break

        G = gradient(W, X, y)
        # TODO C: one full-batch gradient-descent update of W.

    return W, np.arange(0, steps + 1, 5), training_losses

W_trained, iterations, train_losses = train(W_zero, X_train, y_train)

assert np.isfinite(W_trained).all()
assert cross_entropy(X_train, y_train, W_trained) < cross_entropy(X_train, y_train, W_zero)
print('Task C passed — training cross-entropy decreased!')

## 4. Did it work? (about 10 minutes)

**Stop and predict:** What should happen to *training* cross-entropy as gradient descent runs? Is classification accuracy guaranteed to improve at every step?

The plot below tracks **training loss only**. We keep the official test set untouched until the final comparison, so that the lab's numbers can be compared directly with the lecture.

In [ ]:
fig, ax = plt.subplots(figsize=(6.7, 3.2))
ax.plot(iterations, train_losses, marker='o', label='Training cross-entropy')
ax.set(xlabel='Full-batch gradient-descent step', ylabel='Average cross-entropy',
       title='Learning the MNIST linear classifier')
ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.show()

print(f'Training accuracy with class-average weights: {100 * accuracy(X_train, y_train, W_avg):.1f}%')
print(f'Training accuracy with learnt weights:        {100 * accuracy(X_train, y_train, W_trained):.1f}%')

### What did learning change?

The rows of $W$ are still 784-dimensional vectors, so we can view them as images. The class averages contain only nonnegative pixel values. **Learnt weights can also be negative:** a bright pixel at such a location reduces the score for that digit.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 3.5))
limit = max(abs(W_trained.min()), abs(W_trained.max()))
for j in range(10):
    axes[0, j].imshow(W_avg[j].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[1, j].imshow(W_trained[j].reshape(28, 28), cmap='RdBu_r',
                      vmin=-limit, vmax=limit)
    axes[0, j].set_title(str(j))
    axes[0, j].axis('off'); axes[1, j].axis('off')
fig.text(0.005, 0.71, 'Averages', rotation=90, va='center')
fig.text(0.005, 0.29, 'Learnt', rotation=90, va='center')
plt.suptitle('Same score model; different ways to choose W')
plt.tight_layout(rect=[0.025, 0, 1, 0.94]); plt.show()

### Final evaluation on unseen test images

We used **all 60,000 official MNIST training images** to compute class averages and train by gradient descent. Now we evaluate both classifiers on the **same 10,000 official MNIST test images used in the lecture**. Neither model was trained on the test set.

In [ ]:
print(f'Class-average test accuracy: {100 * accuracy(X_test, y_test, W_avg):.1f}%')
print(f'Learnt-weight test accuracy:  {100 * accuracy(X_test, y_test, W_trained):.1f}%')

## Exit questions (last few minutes)

Answer these questions briefly:

1. **Same model?** What changed between the lecture's classifier and today's trained classifier?
2. **Scores vs loss?** Why does cross-entropy depend on the scores for *all ten* digits?
3. **Evidence against a class?** Why can a negative weight be useful?

**Takeaway:** We kept the score function $S(x,j)=(Wx)_j$ and the same full MNIST dataset as in the lecture. Unlike the class-average rule, **gradient descent learnt $W$ by explicitly reducing a training loss**.

*If you finish early:* try changing `learning_rate` to `0.05` or `0.3`. Re-run the training and learning-curve cells. What changes? 